### News

**Read our [Gemma 3 blog](https://unsloth.ai/blog/gemma3) for what's new in Unsloth and our [Reasoning blog](https://unsloth.ai/blog/r1-reasoning) on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [41]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm
# Install latest Hugging Face for Gemma-3!
!pip install --no-deps git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3
!pip install bitsandbytes
!pip install unsloth_zoo
!pip install msgspec
!pip install blake3
!pip install gguf
!pip install networkx==2.8.8 --force-reinstall

In [42]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [43]:
# Force-reinstall Pillow to fix the '_imaging' extension mismatch
!pip install --force-reinstall pillow

# Automatically restart runtime to load the newly installed Pillow version
try:
    from PIL import Image
except ImportError:
    import os
    print("Restarting session to apply Pillow update...")
    os.kill(os.getpid(), 9)

from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

  Using cached pillow-13.0.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached pillow-13.0.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (7.1 MB)
  Attempting uninstall: pillow
    Found existing installation: pillow 13.0.0
    Uninstalling pillow-13.0.0:
      Successfully uninstalled pillow-13.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.33.0 requires fastsafetensors>=0.3.3, which is not installed.
vllm 0.33.0 requires flashinfer-python==0.7.2.post1, which is not installed.
vllm 0.33.0 requires humming-kernels[cu13]==0.1.19, which is not installed.
vllm 0.33.0 requires instanttensor>=0.2.2, which is not installed.
vllm 0.33.0 requires nvidia-cudnn-frontend>=1.31.0, which is not installed.
vllm 0.33.0 requires nvidia-cutlass-dsl[cu13]==4.8.0, which is not installed.
vllm 0.33.0 requires PyN

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

We now add LoRA adapters so we only need to update a small amount of parameters!

In [44]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [45]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [46]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Read the standard CSV cleanly using pandas
df = pd.read_csv("//content/dataasilida_cleaned.csv")

print("Dataset Columns:", df.columns.tolist())
print("Total Rows:", len(df))
print("Sample Input:", df['input'].iloc[0][:50] + "...")
print("Sample Output:", df['output'].iloc[0][:50] + "...")

dataset = DatasetDict({"train": Dataset.from_pandas(df)})

Dataset Columns: ['input', 'output']
Total Rows: 510
Sample Input: តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី?...
Sample Output: ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ ឬកា...


In [47]:
# Print the dataset features and available column names to see what columns exist
print(dataset)
print("Available columns:", dataset['train'].column_names)

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 510
    })
})
Available columns: ['input', 'output']


In [48]:
def formatting_prompts_func(examples):
    instructions = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
      messages = [
        {
            "role": "user", 'content': instruction.strip(" ")
          },
          {
            "role": "assistant", 'content': output
          }
      ]
      #tokenizer.apply_chat_template(messages, add_generation_prompt=False , tokenize= False)
      text = tokenizer.apply_chat_template(messages, add_generation_prompt=False , tokenize= False)
      texts.append(text)
    return {"text":texts}



In [49]:
# Map the formatting function directly using our correctly parsed 'dataset' variable
dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/510 [00:00<?, ? examples/s]

In [50]:
dataset['train']['text'][0]

'<bos><start_of_turn>user\nតើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី?<end_of_turn>\n<start_of_turn>model\nប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ ឬកាតអេឡិចត្រូនិក បោះផ្សាយដោយ ធនាគារ អេស៊ីលីដា ភីអិលស៊ី សម្រាប់ផ្តល់ជូនអតិថិជនក្នុងការធ្វើប្រតិបត្តិការតាមរយៈ E-Commerce ម៉ាស៊ីនឆូតកាត (POS) ម៉ាស៊ីនអេធីអឹម (ATM) ដែលមានបិទស្លាកសញ្ញា ចេស៊ីប៊ី នៅទូទាំងប្រទេស និងពាសពេញពិភពលោក ដោយធ្វើការទូទាត់សាច់ប្រាក់ចេញពីគណនីរបស់ម្ចាស់ប័ណ្ណដែលមាននៅធនាគារ អេស៊ីលីដា។<end_of_turn>\n'

Let's see how row 100 looks like!

In [54]:
dataset['train']['input'][50]

'ករណីខ្ញុំជ្រើសប្រើ ACLEDA Authentication តើត្រូវធ្វើដូចម្តេច'

In [52]:
dataset['train']['input'][0]

'តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី?'

Let's see how the chat template did! Notice `Gemma-3` default adds a `<bos>`!

In [53]:
dataset['train']['output'][00]

'ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ ឬកាតអេឡិចត្រូនិក បោះផ្សាយដោយ ធនាគារ អេស៊ីលីដា ភីអិលស៊ី សម្រាប់ផ្តល់ជូនអតិថិជនក្នុងការធ្វើប្រតិបត្តិការតាមរយៈ E-Commerce ម៉ាស៊ីនឆូតកាត (POS) ម៉ាស៊ីនអេធីអឹម (ATM) ដែលមានបិទស្លាកសញ្ញា ចេស៊ីប៊ី នៅទូទាំងប្រទេស និងពាសពេញពិភពលោក ដោយធ្វើការទូទាត់សាច់ប្រាក់ចេញពីគណនីរបស់ម្ចាស់ប័ណ្ណដែលមាននៅធនាគារ អេស៊ីលីដា។'

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [14]:
# from trl import SFTTrainer, SFTConfig
# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = dataset,
#     eval_dataset = None, # Can set up evaluation!
#     args = SFTConfig(
#         dataset_text_field = "text",
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4, # Use GA to mimic batch size!
#         warmup_steps = 5,
#         # num_train_epochs = 1, # Set this for 1 full training run.
#         max_steps = 60,
#         learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
#         logging_steps = 1,
#         optim = "adamw_8bit",
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
#         seed = 3407,
#         report_to = "none", # Use this for WandB etc
#     ),
# )

In [15]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    dataset_text_field = "text",
    #max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/510 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [16]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map:   0%|          | 0/510 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again:

In [17]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<bos><start_of_turn>user\nដូចម្តេចដែលហៅថាសេវាបញ្ជាទូទាត់តាមរយៈ E-mail/Fax?<end_of_turn>\n<start_of_turn>model\nសេវាបញ្ជាទូទាត់តាមរយៈ E-mail / Fax គឺជាប្រតិបត្តិការផ្ទេរប្រាក់ ឬដកប្រាក់ពីគណនី តាមការណែនាំរបស់ម្ចាស់គណនី ដែលមានជាមួយធនាគារ អេស៊ីលីដា ហើយបានផ្ញើ E-mail / Fax មកលោកនាយកប្រតិបត្តិការ ឬនាយក-នាយិកាសាខា នៃធនាគារ អេស៊ីលីដា ដើម្បីធ្វើប្រតិបត្តិការទូទាត់តាមបំណងរបស់ខ្លួន។<end_of_turn>\n'

Now let's print the masked out example - you should see only the answer is present:

In [18]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                 សេវាបញ្ជាទូទាត់តាមរយៈ E-mail / Fax គឺជាប្រតិបត្តិការផ្ទេរប្រាក់ ឬដកប្រាក់ពីគណនី តាមការណែនាំរបស់ម្ចាស់គណនី ដែលមានជាមួយធនាគារ អេស៊ីលីដា ហើយបានផ្ញើ E-mail / Fax មកលោកនាយកប្រតិបត្តិការ ឬនាយក-នាយិកាសាខា នៃធនាគារ អេស៊ីលីដា ដើម្បីធ្វើប្រតិបត្តិការទូទាត់តាមបំណងរបស់ខ្លួន។<end_of_turn> '

In [19]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
4.381 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [20]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 106}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 510 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248 of 4,314,980,720 (0.35% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,3.749756
2,3.030968
3,3.111783
4,3.373418
5,2.937788
6,3.034578
7,2.050462
8,2.148847
9,2.053454
10,1.939974


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


### Custom Model Inference
Use the cell below to write custom prompt inputs and obtain predictions from your newly finetuned model.

In [21]:
from transformers import TextStreamer

# Input your custom prompt here
user_input = "តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី?"

messages = [{
    "role": "user",
    "content": [{"type": "text", "text": user_input}]
}]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
)

print("Generating response...")
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 256,
    temperature = 1.0,
    top_p = 0.95,
    top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Generating response...
ប័ណ្ណឥណពន្ធសាច់ប្រាក់សាមញ្ញ អេស៊ីលីដា សំរាប់អ្នកគ្រូបង្រួញ បម្រើគ្រប់មុខរបរ នៅតាមហាងទំនិញ ដែលសំដៅជាហាងទំនិញ និងភោជនីការដ្ឋាន ស្ថិតនៅតាមការគិតទៅតាមពូបាលៃអត្រាអចិនតេត ដែលសមរម្យ និងពិតប្រាកដតាមពូបាលៃដែលជារូបភាពត្រប់ អត្រាពន្ធប្រាក់ការទូទាត់ដែលនឹងប្រទូបទាត់ការទូទាត់ ក្នុងនោះអ្នកទូទាត់ ឬអ្នកទទួលភារកិច្គេបងបែកសាច់ប្រាក់ពីលាយភិនរបស់ខ្លួនឯង អំងាយជាមានពិតប្រាកដ អត្រាប្រាក់ការទូទាត់ អត្រាប្រាក់ការទូទាត់។<end_of_turn>


In [22]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

304.9859 seconds used for training.
5.08 minutes used for training.
Peak reserved memory = 5.1 GB.
Peak reserved memory for training = 0.719 GB.
Peak reserved memory % of max memory = 35.02 %.
Peak reserved memory for training % of max memory = 4.937 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [23]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "ទៅមុខ១០មហើយបត់ឆ្វេង",
    }]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)
outputs = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

['<bos><start_of_turn>user\nទៅមុខ១០មហើយបត់ឆ្វេង<end_of_turn>\n<start_of_turn>model\nបត់ឆ្វេង ទៅមុខ១០ម ហើយបន្តទៅមុខទៀត។<end_of_turn>']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [24]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "ទៅមុខ១០ម",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Please provide me with the initial text or context in which to write a short tale, in order to write ទៅមុខ១០ម.<end_of_turn>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [25]:
model.save_pretrained("gemma-3")  # Local saving
tokenizer.save_pretrained("gemma-3")
# model.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving

Unsloth: Restored added_tokens_decoder metadata in gemma-3/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma-3.


['gemma-3/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [26]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "បន្តទៅក្រោយ៩ម",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

ដក tilbake ប្រសិនគិតក្នុងបանկអាយុពេលទម្រាំដល់ទៅ ១៥៥៤១ អាចដាក់ចេញសាច់ប្រាក់ពីមុខរបរប៉ុន្តែក្ដីអោយលោកអ្នកគិតប្រាក់ត្រឡប់មកខាងក្រោមទើបដំណើរការ។ 


### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-3-finetune`. Set `if False` to `if True` to let it run!

In [27]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-3-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [28]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-3-finetune", tokenizer,
        token = "hf_..."
    )

In [29]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [30]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # Only Q8_0, BF16, F16 supported
        repo_id = "HF_ACCOUNT/gemma-finetune-gguf",
        token = "hf_...",
    )

In [31]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/gemma-3",
    "zip",
    root_dir="/content",
    base_dir="gemma-3",
)
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:
from pathlib import Path
from zipfile import ZipFile
import shutil
from getpass import getpass
from google.colab import files
from unsloth import FastModel

# Upload the gemma-3.zip file that contains adapter_config.json.
uploaded = files.upload()
zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_files:
    raise ValueError("Please upload the gemma-3.zip adapter file.")

# Extract the adapter and verify that this is the expected folder.
extract_dir = Path("/content/uploaded_gemma_adapter")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)
with ZipFile(zip_files[0]) as archive:
    root = extract_dir.resolve()
    for member in archive.infolist():
        destination = (extract_dir / member.filename).resolve()
        if destination != root and root not in destination.parents:
            raise ValueError("The zip contains a path outside the adapter folder.")
    archive.extractall(extract_dir)

config_files = list(extract_dir.rglob("adapter_config.json"))
if not config_files:
    raise FileNotFoundError("adapter_config.json was not found in the uploaded zip.")
adapter_dir = config_files[0].parent
print("Adapter folder:", adapter_dir)

# Enter a Hugging Face token only if the model repository asks for one.
hf_token = getpass("Hugging Face token (press Enter to skip): ").strip() or None

# Load the base Gemma model and attach the saved LoRA adapter.
model, tokenizer = FastModel.from_pretrained(
    model_name=str(adapter_dir),
    max_seq_length=2048,
    load_in_4bit=True,
    token=hf_token,
)
model.eval()
print("Adapter and base model are ready. Run the Gradio cell below.")


Saving gemma-3.zip to gemma-3 (1).zip
Adapter folder: /content/uploaded_gemma_adapter/gemma-3
Hugging Face token (press Enter to skip): ··········
==((====))==  Unsloth 2026.11.4: Fast Gemma3 patching. Transformers: 5.22.2. vLLM: 0.33.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

In [40]:
!pip -q install gradio

import gradio as gr
import torch

model.eval()

def gemma_chat(message, history, max_new_tokens, temperature):
    # Rebuild the earlier turns so the model can follow the conversation.
    messages = []
    for item in history:
        if isinstance(item, list) and len(item) == 2:
            user_text, assistant_text = item
            messages.append({"role": "user", "content": user_text})
            messages.append({"role": "assistant", "content": assistant_text})

    messages.append({"role": "user", "content": message})

    # Format the conversation the way Gemma expects it.
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to("cuda")

    prompt_length = inputs["input_ids"].shape[-1]

    # Choose how the model generates its answer.
    generation_options = {
        "max_new_tokens": int(max_new_tokens),
    }
    if temperature > 0:
        generation_options.update(
            do_sample=True,
            temperature=float(temperature),
            top_p=0.95,
            top_k=64,
        )
    else:
        generation_options["do_sample"] = False

    with torch.inference_mode():
        output = model.generate(**inputs, **generation_options)

    answer_tokens = output[0, prompt_length:]
    return tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True,
    ).strip()


demo = gr.ChatInterface(
    fn=gemma_chat,
    title="Gemma 3 Fine-Tune Tester",
    description="Ask a question and try the fine-tuned model.",
    examples=[
        ["តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី?", 256, 0.7],
        ["ទៅមុខ១០មហើយបត់ឆ្វេង", 256, 0.7],
    ],
    additional_inputs=[
        gr.Slider(
            32, 512, value=256, step=32,
            label="Maximum response length",
        ),
        gr.Slider(
            0, 1.5, value=0.7, step=0.1,
            label="Temperature",
        ),
    ],
)

demo.queue().launch(share=True, show_error=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://41864eac0a03c50402.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
